In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import torch
from datasets import load_dataset
from PIL import Image
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch import nn, optim
from torch.utils.data import DataLoader, Dataset
from torchinfo import summary
from torchvision import transforms
from tqdm import tqdm

1.8.0


In [ ]:
class SimpleCNN(nn.Module):
    """Simple CNN for binary classification of cats vs dogs."""

    def __init__(self, num_classes=2):
        super(SimpleCNN, self).__init__()

        # Convolutional Block 1
        # Input: (batch_size, 3, H, W) - RGB images
        # Output: (batch_size, 32, H/2, W/2) - 32 feature maps, half the spatial size
        self.conv_block1 = nn.Sequential(
            # First conv: 3 input channels (RGB) → 32 output feature maps
            # kernel_size=3: uses 3x3 filters, padding=1: maintains spatial dimensions
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),  # Normalizes the 32 feature maps for stable training
            nn.ReLU(),  # Non-linear activation function
            # Second conv: 32 → 32 feature maps, learns more complex patterns
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            # Downsample by 2x: reduces spatial dimensions (e.g., 224x224 → 112x112)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 2
        # Input: (batch_size, 32, H/2, W/2)
        # Output: (batch_size, 64, H/4, W/4) - doubles feature maps, halves spatial size
        self.conv_block2 = nn.Sequential(
            # Increase depth: 32 → 64 feature maps for learning more complex features
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            # Downsample again: (e.g., 112x112 → 56x56)
            nn.MaxPool2d(kernel_size=2, stride=2),
        )

        # Convolutional Block 3
        # Input: (batch_size, 64, H/4, W/4)
        # Output: (batch_size, 128, 4, 4) - highest-level features, fixed 4x4 spatial size
        self.conv_block3 = nn.Sequential(
            # Increase depth: 64 → 128 feature maps for high-level feature learning
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Second conv at this depth level
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # Final spatial downsampling (e.g., 56x56 → 28x28)
            nn.MaxPool2d(kernel_size=2, stride=2),
            # Adaptive pooling: converts any spatial size to fixed 4x4
            # This makes the network flexible to different input image sizes
            nn.AdaptiveAvgPool2d((4, 4)),
        )

        # Fully Connected Layers (Classifier)
        # Input: (batch_size, 128, 4, 4) = flattened to (batch_size, 2048)
        # Output: (batch_size, 2) - raw scores for cat and dog classes
        self.classifier = nn.Sequential(
            # Flatten 3D feature maps into 1D vector: 128×4×4 = 2048 features
            nn.Flatten(),
            # First dense layer: 2048 → 512 neurons
            nn.Linear(128 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),  # Randomly drop 50% of neurons during training to prevent overfitting
            # Second dense layer: 512 → 256 neurons
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.5),  # Another dropout layer for regularization
            # Output layer: 256 → 2 class scores (logits for cat and dog)
            nn.Linear(256, 2),
        )

    def forward(self, x):
        """
        Forward pass through the network.

        Args:
            x: Input tensor of shape (batch_size, 3, H, W)

        Returns:
            Output tensor of shape (batch_size, 2) containing class logits
        """
        # Pass through convolutional blocks to extract features
        x = self.conv_block1(x)  # Extract low-level features (edges, textures)
        x = self.conv_block2(x)  # Extract mid-level features (shapes, patterns)
        x = self.conv_block3(x)  # Extract high-level features (animal parts, structures)

        # Pass through classifier to get final predictions
        x = self.classifier(x)  # Convert features to class scores

        return x  # Returns raw logits (use with CrossEntropyLoss or apply softmax for probabilities)

In [28]:
class CatsDogsDataset(Dataset):
    """PyTorch Dataset for Cats vs Dogs."""

    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        image = self.images[idx]
        label = self.labels[idx]

        # Convert numpy array to PIL Image if needed
        if isinstance(image, np.ndarray):
            image = Image.fromarray(image.astype("uint8"))

        if self.transform:
            image = self.transform(image)

        return image, label

In [29]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [30]:
def validate(model, dataloader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [31]:
def get_predictions(model, dataloader, device):
    """Get all predictions and true labels."""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)

In [32]:
def plot_training_history(history):
    """Plot training and validation loss and accuracy."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Plot loss
    ax1.plot(history["train_loss"], label="Train Loss", marker="o", linewidth=2)
    ax1.plot(history["val_loss"], label="Validation Loss", marker="o", linewidth=2)
    ax1.set_xlabel("Epoch", fontsize=12)
    ax1.set_ylabel("Loss", fontsize=12)
    ax1.set_title("Training and Validation Loss", fontsize=14, fontweight="bold")
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # Plot accuracy
    ax2.plot(history["train_acc"], label="Train Accuracy", marker="o", linewidth=2)
    ax2.plot(history["val_acc"], label="Validation Accuracy", marker="o", linewidth=2)
    ax2.set_xlabel("Epoch", fontsize=12)
    ax2.set_ylabel("Accuracy (%)", fontsize=12)
    ax2.set_title("Training and Validation Accuracy", fontsize=14, fontweight="bold")
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig("training_history.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Training history plot saved as 'training_history.png'\n")

In [33]:
def plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Plot confusion matrix using only matplotlib."""
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(8, 6))

    # Create heatmap using imshow
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")

    # Add colorbar
    cbar = plt.colorbar(im, ax=ax)
    cbar.set_label("Count", rotation=270, labelpad=20)

    # Set ticks and labels
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes)
    ax.set_yticklabels(classes)

    # Rotate the tick labels for better readability
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

    # Add text annotations
    thresh = cm.max() / 2.0
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(
                j,
                i,
                format(cm[i, j], "d"),
                ha="center",
                va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontsize=14,
            )

    ax.set_xlabel("Predicted Label", fontsize=12, fontweight="bold")
    ax.set_ylabel("True Label", fontsize=12, fontweight="bold")
    ax.set_title("Confusion Matrix", fontsize=14, fontweight="bold")

    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
    plt.show()
    print("✓ Confusion matrix saved as 'confusion_matrix.png'\n")

In [34]:
def print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"]):
    """Print and save classification report."""
    print("=" * 80)
    print("CLASSIFICATION REPORT ON TEST DATASET")
    print("=" * 80)

    report = classification_report(y_true, y_pred, target_names=classes, digits=4)
    print(report)

    # Save to file
    with open("classification_report.txt", "w") as f:
        f.write("CLASSIFICATION REPORT ON TEST DATASET\n")
        f.write("=" * 80 + "\n")
        f.write(report)

    print("=" * 80)
    print("✓ Classification report saved as 'classification_report.txt'\n")

In [ ]:
class FeatureExtractor:
    """Extract feature vectors from CNN model."""

    def __init__(self, model, device, layer_name="classifier"):
        """
        Initialize feature extractor.

        Args:
            model: Trained CNN model
            device: torch device
            layer_name: Name of layer to extract features from
        """
        self.model = model
        self.device = device
        self.model.eval()

        # Hook to extract features from specific layer
        self.features = None

        # Extract features before final classification layer
        if layer_name == "classifier":
            # Get the layer before the final linear layer
            target_layer = self.model.classifier[-2]  # Before final Linear layer
        else:
            target_layer = getattr(self.model, layer_name)

        target_layer.register_forward_hook(self._hook_fn)

    def _hook_fn(self, module, input, output):
        """Hook function to capture layer output."""
        self.features = output.detach()

    def extract(self, image_tensor):
        """
        Extract feature vector from image.

        Args:
            image_tensor: Preprocessed image tensor (1, C, H, W)

        Returns:
            Feature vector as numpy array
        """
        with torch.no_grad():
            _ = self.model(image_tensor.to(self.device))
            features = self.features.cpu().numpy().flatten()
        return features


def build_vector_database(model, dataloader, device, collection_name="image_features"):
    """
    Build vector database from image dataset.

    Args:
        model: Trained CNN model
        dataloader: DataLoader with images
        device: torch device
        collection_name: Name for Qdrant collection

    Returns:
        Tuple of (qdrant_client, feature_extractor, image_metadata)
    """
    print("\n" + "=" * 80)
    print("BUILDING SIMILARITY SEARCH ENGINE")
    print("=" * 80)

    # Initialize feature extractor
    print("\n1. Initializing feature extractor...")
    feature_extractor = FeatureExtractor(model, device)

    # Get feature dimension
    sample_batch = next(iter(dataloader))
    sample_features = feature_extractor.extract(sample_batch[0][0:1])
    feature_dim = len(sample_features)
    print(f"   ✓ Feature dimension: {feature_dim}")

    # Initialize Qdrant client 
    print("\n2. Initializing Qdrant vector database...")
    client = QdrantClient(":memory:")  # In-memory database

    # Create collection
    client.create_collection(
        collection_name=collection_name, vectors_config=VectorParams(size=feature_dim, distance=Distance.COSINE)
    )
    print(f"   Created collection '{collection_name}'")

    # Extract features and populate database
    print("\n3. Extracting features and populating database...")

    points = []
    image_metadata = []
    point_id = 0

    for images, labels in tqdm(dataloader, desc="Processing images"):
        for i in range(images.size(0)):
            # Extract features
            features = feature_extractor.extract(images[i : i + 1])

            # Store metadata
            metadata = {"id": point_id, "label": int(labels[i].item()), "image_tensor": images[i].cpu()}
            image_metadata.append(metadata)

            # Create point for Qdrant
            point = PointStruct(id=point_id, vector=features.tolist(), payload={"label": int(labels[i].item())})
            points.append(point)
            point_id += 1

    # Upload to Qdrant in batches
    batch_size = 100
    for i in range(0, len(points), batch_size):
        client.upsert(collection_name=collection_name, points=points[i : i + batch_size])

    print(f"   Indexed {len(points)} images")

    # Verify collection
    collection_info = client.get_collection(collection_name)
    print("\nDatabase:")
    print(f"   - Total vectors: {collection_info.points_count}")
    print(f"   - Vector dimension: {feature_dim}")
    print("   - Distance metric: Cosine similarity")

    return client, feature_extractor, image_metadata


def search_similar_images(client, feature_extractor, query_image, k=5, collection_name="image_features"):
    """
    Search for k most similar images.

    Args:
        client: Qdrant client
        feature_extractor: Feature extractor instance
        query_image: Query image tensor (1, C, H, W)
        k: Number of similar images to retrieve
        collection_name: Qdrant collection name

    Returns:
        List of search results with scores
    """
    # Extract features from query image
    query_features = feature_extractor.extract(query_image)

    # Search in Qdrant
    search_results = client.search(collection_name=collection_name, query_vector=query_features.tolist(), limit=k)

    return search_results


def visualize_search_results(query_image, query_label, search_results, image_metadata, class_names=["Cat", "Dog"]):
    """
    Visualize query image and retrieved similar images.

    Args:
        query_image: Query image tensor
        query_label: True label of query image
        search_results: Results from Qdrant search
        image_metadata: Metadata containing image tensors
        class_names: List of class names
    """
    # Denormalization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    def denormalize(img):
        img = img * std + mean
        img = torch.clamp(img, 0, 1)
        return img.permute(1, 2, 0).numpy()

    num_results = len(search_results)
    fig, axes = plt.subplots(1, num_results + 1, figsize=(3 * (num_results + 1), 3))

    # Plot query image
    query_img = denormalize(query_image.cpu())
    axes[0].imshow(query_img)
    axes[0].set_title(f"Query Image\n{class_names[query_label]}", fontsize=12, fontweight="bold", color="blue")
    axes[0].axis("off")
    axes[0].add_patch(
        plt.Rectangle((0, 0), query_img.shape[1], query_img.shape[0], fill=False, edgecolor="blue", linewidth=3)
    )

    # Plot retrieved images
    for idx, result in enumerate(search_results):
        # Get image from metadata
        img_id = result.id
        img_meta = image_metadata[img_id]
        img_tensor = img_meta["image_tensor"]
        img_label = img_meta["label"]
        similarity_score = result.score

        # Display image
        img = denormalize(img_tensor)
        axes[idx + 1].imshow(img)

        # Color code based on correctness
        is_correct = img_label == query_label
        color = "green" if is_correct else "red"

        axes[idx + 1].set_title(
            f"Rank {idx + 1}\n{class_names[img_label]}\nScore: {similarity_score:.4f}",
            fontsize=10,
            color=color,
            fontweight="bold" if idx == 0 else "normal",
        )
        axes[idx + 1].axis("off")

    plt.tight_layout()
    return fig


def run_similarity_search_demo(model, test_loader, device, k=5, num_queries=3, class_names=["Cat", "Dog"]):
    """
    Complete similarity search demonstration.

    Args:
        model: Trained CNN model
        test_loader: Test data loader
        device: torch device
        k: Number of similar images to retrieve
        num_queries: Number of query images to test
        class_names: List of class names
    """
    print("\n" + "=" * 80)
    print("TASK 4: CNN-BASED SIMILARITY SEARCH ENGINE")
    print("=" * 80)

    # Build vector database
    client, feature_extractor, image_metadata = build_vector_database(model, test_loader, device)

    # Select random query images
    print("\n" + "=" * 80)
    print(f"PERFORMING SIMILARITY SEARCH (k={k})")
    print("=" * 80)

    # Get random images from test set
    all_images, all_labels = next(iter(test_loader))

    # Select num_queries random indices
    np.random.seed(42)
    query_indices = np.random.choice(len(all_images), size=num_queries, replace=False)

    results_summary = []

    for query_idx_local, query_idx in enumerate(query_indices):
        query_image = all_images[query_idx : query_idx + 1]
        query_label = all_labels[query_idx].item()

        print(f"\n{'=' * 80}")
        print(f"Query Image {query_idx_local + 1}/{num_queries}")
        print(f"{'=' * 80}")
        print(f"Query: {class_names[query_label]}")

        # Search for similar images
        search_results = search_similar_images(client, feature_extractor, query_image, k=k)

        print(f"\nTop-{k} Similar Images:")
        print(f"{'Rank':<6} {'Class':<10} {'Similarity Score':<20} {'Match'}")
        print("-" * 80)

        for idx, result in enumerate(search_results):
            img_meta = image_metadata[result.id]
            img_label = img_meta["label"]
            similarity_score = result.score
            is_match = "✓" if img_label == query_label else "✗"

            print(f"{idx + 1:<6} {class_names[img_label]:<10} {similarity_score:<20.4f} {is_match}")

        # Calculate precision@k
        correct_matches = sum(1 for r in search_results if image_metadata[r.id]["label"] == query_label)
        precision_at_k = correct_matches / k
        print(f"\nPrecision@{k}: {precision_at_k:.2%} ({correct_matches}/{k} correct)")

        # Visualize results
        fig = visualize_search_results(query_image[0], query_label, search_results, image_metadata, class_names)

        # Save figure
        filename = f"similarity_search_query_{query_idx_local + 1}.png"
        plt.savefig(filename, dpi=150, bbox_inches="tight")
        print(f"✓ Visualization saved as '{filename}'")
        plt.show()

        results_summary.append(
            {
                "query_class": class_names[query_label],
                "precision_at_k": precision_at_k,
                "top_scores": [r.score for r in search_results],
            }
        )

    # Print summary
    print("\n" + "=" * 80)
    print("SIMILARITY SEARCH SUMMARY")
    print("=" * 80)

    avg_precision = np.mean([r["precision_at_k"] for r in results_summary])
    print(f"\nAverage Precision@{k}: {avg_precision:.2%}")

    return client, feature_extractor, results_summary


In [ ]:
# Hyperparameters
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 0.001
NUM_CLASSES = 2

# Control flags
FORCE_RETRAIN = False  # Set to True to retrain even if model exists
MODEL_PATH = "best_model.pth"
SKIP_DATA_CACHE = True  # Set to False if you want to cache data (slower first run, faster later)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}\n")


# ============================================================================
# HELPER: Check if we should skip training
# ============================================================================
def should_skip_training():
    """Check if model exists and we shouldn't force retrain."""
    if FORCE_RETRAIN:
        print("⚠️ FORCE_RETRAIN is True - will retrain model")
        return False

    if os.path.exists(MODEL_PATH):
        print(f"✓ Found existing model at '{MODEL_PATH}'")
        print("  Set FORCE_RETRAIN=True to retrain")
        return True

    print("✗ No existing model found - will train")
    return False


# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("=" * 80)
print("DATA LOADING")
print("=" * 80)

# For simplicity, always load data fresh (PIL images can't be easily cached)
print("Loading dataset")
data = load_dataset("pantelism/cats-vs-dogs")

print("Preparing images...")
train_images = [img for img in data["train"]["image"]]
train_labels = np.array(data["train"]["label"])

print(f"Total images: {len(train_images)}")
print(f"Label distribution: {np.unique(train_labels, return_counts=True)}\n")

# Split into train, validation, and test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    train_images, train_labels, test_size=0.3, random_state=42, stratify=train_labels
)

val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42, stratify=temp_lbls
)

print(f"\nTrain set: {len(train_imgs)} images")
print(f"Validation set: {len(val_imgs)} images")
print(f"Test set: {len(test_imgs)} images\n")

# ============================================================================
# 2. CREATE DATASETS AND DATALOADERS
# ============================================================================
print("Creating datasets...")

# Data transforms
train_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

val_transform = transforms.Compose(
    [
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ]
)

# Create datasets and dataloaders
train_dataset = CatsDogsDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset = CatsDogsDataset(val_imgs, val_lbls, transform=val_transform)
test_dataset = CatsDogsDataset(test_imgs, test_lbls, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print("✓ Datasets ready!\n")

# ============================================================================
# 3. CREATE MODEL
# ============================================================================
print("=" * 80)
print("MODEL SETUP")
print("=" * 80)

model = SimpleCNN(num_classes=NUM_CLASSES)
model = model.to(device)

# Print model summary
print(model)
print("\n" + "=" * 80)
print("MODEL SUMMARY")
print("=" * 80)
print(
    summary(
        model,
        input_size=(BATCH_SIZE, 3, IMG_SIZE, IMG_SIZE),
        col_names=["input_size", "output_size", "num_params"],
        depth=3,
    )
)
print("=" * 80 + "\n")

# ============================================================================
# 4. TRAIN THE MODEL (OR SKIP IF MODEL EXISTS)
# ============================================================================
if should_skip_training():
    print("\n⏭️  SKIPPING TRAINING - Loading existing model\n")
    model.load_state_dict(torch.load(MODEL_PATH))
    print(f"✓ Model loaded from '{MODEL_PATH}'\n")

else:
    print("\n" + "=" * 80)
    print("TRAINING MODEL")
    print("=" * 80)

    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3)

    print("Starting training...\n")
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    best_val_acc = 0.0

    for epoch in range(EPOCHS):
        print(f"Epoch {epoch + 1}/{EPOCHS}")
        print("-" * 40)

        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        history["train_loss"].append(train_loss)
        history["train_acc"].append(train_acc)
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

        # Validate
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        # Learning rate scheduling
        scheduler.step(val_loss)

        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), MODEL_PATH)
            print("✓ Model saved!")
        print()

    print(f"Training complete! Best validation accuracy: {best_val_acc:.2f}%\n")

    # Plot training history
    plot_training_history(history)

    # Load best model
    model.load_state_dict(torch.load(MODEL_PATH))

# ============================================================================
# 5. EVALUATE ON TEST SET
# ============================================================================
print("\n" + "=" * 80)
print("TEST SET EVALUATION")
print("=" * 80)

criterion = nn.CrossEntropyLoss()

print("Evaluating on test set...")
test_loss, test_acc = validate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.2%}\n")

# Confusion Matrix
print("Generating predictions...")
y_pred, y_true = get_predictions(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# Classification Report
print_classification_report(y_true, y_pred, classes=["Cat (0)", "Dog (1)"])

# ========================================================================
# SUMMARY
# ========================================================================

print(f"\nFinal Test Accuracy: {test_acc:.2f}%")
print("=" * 80)

print("\\n" + "=" * 80)
print("TASK 4: SIMILARITY SEARCH ENGINE")
print("=" * 80)

# Load best model
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# Run similarity search demo
# k = number of similar images to retrieve
k = 5  # or 3, 7, 10, etc.

client, feature_extractor, results = run_similarity_search_demo(
    model,
    test_loader,
    device,
    k=k,
    num_queries=3,  # Test with 3 random images
    class_names=["Cat", "Dog"],
)

print("\\n" + "=" * 80)
print("ALL TASKS COMPLETE!")
print("=" * 80)